# MM-CLF-CBF Safe Sampling for 2D Obstacle Avoidance

Parallel of `mnist-cbf-clf-v2.ipynb` for a 2D trajectory-generation task. Same QP machinery, different problem:

- A flow model is trained to generate **planar trajectories** $\tau = (z_1, z_2, \dots, z_T) \in \mathbb{R}^{T \times 2}$ from a start region to a goal region.
- At test time, **obstacles** appear (one or more circles) and we want the generated trajectories to avoid them.
- The barrier is geometric and exact (no classifier training needed):

$$h(\tau) = \min_{t,\,o} \left( \|z_t - c_o\|^2 - (r_o + m)^2 \right)$$

approximated by a soft-min over waypoints and obstacles so $\nabla_\tau h$ is smooth.

**Methods compared (same as MNIST notebook):**
1. **Nominal** flow (no obstacle awareness)
2. **CBF** half-space projection
3. **CBF + loglike-CLF** (likelihood preservation via continuity equation)
4. **CBF + relu-CLF** (ablation, expected redundant)
5. **Classifier guidance** baseline — gradient of obstacle penalty added to velocity

**Why obstacles are cleaner than MNIST:** $h$ is geometric and has an *exact closed form* (no trained classifier, no kNN, no Tweedie estimate). This isolates the CBF/CLF formulation from barrier-quality issues.

In [ ]:
%pip install torch torchcfm tqdm -q

import os, math
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchcfm.conditional_flow_matching import ConditionalFlowMatcher
from tqdm import tqdm

plt.rcParams["figure.figsize"] = [10, 4]
plt.rcParams["figure.dpi"] = 100

if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print("device:", device)

torch.manual_seed(0)
np.random.seed(0)

## 1. Training distribution: clean trajectories

Trajectories start near $(-3, 0)$, end near $(3, 0)$, $T=32$ waypoints. We sample smooth random curves (cubic Bézier control points + noise) so the *unconditional* flow learns a rich distribution of plausible paths between start and goal. **No obstacles seen at training time** — the obstacle avoidance is purely a test-time CBF concern.

In [ ]:
T = 32                # waypoints per trajectory
D = T * 2             # flat dim
START = torch.tensor([-3.0, 0.0])
GOAL  = torch.tensor([ 3.0, 0.0])

def sample_trajectories(n, T=T, noise=0.15):
    """Smooth random curves from START to GOAL via two random control points."""
    s = torch.linspace(0, 1, T).view(1, T, 1)            # (1, T, 1)
    p0 = START.view(1, 1, 2).expand(n, 1, 2)
    p3 = GOAL.view(1, 1, 2).expand(n, 1, 2)
    # control points in a band around the straight line
    p1 = torch.stack([torch.empty(n).uniform_(-2, 0),
                       torch.empty(n).uniform_(-2.5, 2.5)], dim=1).unsqueeze(1)
    p2 = torch.stack([torch.empty(n).uniform_(0, 2),
                       torch.empty(n).uniform_(-2.5, 2.5)], dim=1).unsqueeze(1)
    # cubic Bezier
    b = ((1 - s) ** 3) * p0 + 3 * ((1 - s) ** 2) * s * p1 + 3 * (1 - s) * (s ** 2) * p2 + (s ** 3) * p3
    b = b + noise * torch.randn_like(b)
    return b                                              # (n, T, 2)

trajs = sample_trajectories(3000)
print("trajs shape:", trajs.shape)

# Visualise a few training trajectories
fig, ax = plt.subplots(figsize=(6, 5))
for k in range(50):
    ax.plot(trajs[k, :, 0], trajs[k, :, 1], alpha=0.3, color="steelblue")
ax.plot(*START, "go", ms=12, label="start")
ax.plot(*GOAL,  "r*", ms=14, label="goal")
ax.set_aspect("equal"); ax.legend(); ax.set_title("Training trajectories (no obstacles)")
plt.tight_layout(); plt.show()

## 2. Flow matching model

Small MLP $v_\theta(t, \tau)$ that takes flattened trajectory $\tau \in \mathbb{R}^{64}$ and time $t$, predicts velocity in $\mathbb{R}^{64}$. Standard linear-interpolant CFM with $\sigma=0$.

In [ ]:
class TimeMLP(nn.Module):
    def __init__(self, d=D, hidden=256, n_layers=4):
        super().__init__()
        layers = [nn.Linear(d + 1, hidden), nn.SiLU()]
        for _ in range(n_layers - 1):
            layers += [nn.Linear(hidden, hidden), nn.SiLU()]
        layers += [nn.Linear(hidden, d)]
        self.net = nn.Sequential(*layers)
        self.d = d

    def forward(self, t, x):
        # x: (B, T, 2)  →  (B, d)
        if x.ndim == 3:
            x_flat = x.reshape(x.size(0), -1)
        else:
            x_flat = x
        if t.ndim == 0:
            t = t.expand(x_flat.size(0))
        out = self.net(torch.cat([x_flat, t.view(-1, 1)], dim=1))  # (B, d)
        return out.view_as(x)

flow_model = TimeMLP().to(device)
optimizer  = optim.Adam(flow_model.parameters(), lr=1e-3)
FM = ConditionalFlowMatcher(sigma=0.0)

n_epochs = 200
batch    = 256
trajs_dev = trajs.to(device)

flow_model.train()
for epoch in range(n_epochs):
    perm = torch.randperm(trajs_dev.size(0), device=device)
    total = 0.0; n = 0
    for i in range(0, trajs_dev.size(0), batch):
        x1 = trajs_dev[perm[i:i+batch]]
        x0 = torch.randn_like(x1)
        t, xt, ut = FM.sample_location_and_conditional_flow(x0, x1)
        vt = flow_model(t, xt)
        loss = ((vt - ut) ** 2).mean()
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        total += loss.item() * x1.size(0); n += x1.size(0)
    if (epoch + 1) % 25 == 0:
        print(f"epoch {epoch+1:3d}  loss {total/n:.4f}")

In [ ]:
## Visualise unconditional samples
@torch.no_grad()
def euler_sample(flow_model, x0, num_steps=100):
    x = x0.clone(); dt = 1.0 / num_steps
    for k in range(num_steps):
        t_in = torch.full((x.size(0),), k * dt, device=x.device)
        x = x + dt * flow_model(t_in, x)
    return x

flow_model.eval()
x0 = torch.randn(40, T, 2, device=device)
x_gen = euler_sample(flow_model, x0)

fig, ax = plt.subplots(figsize=(6, 5))
for k in range(40):
    ax.plot(x_gen[k, :, 0].cpu(), x_gen[k, :, 1].cpu(), alpha=0.4, color="steelblue")
ax.plot(*START, "go", ms=12); ax.plot(*GOAL, "r*", ms=14)
ax.set_aspect("equal"); ax.set_title("Nominal generated trajectories (no obstacles enforced)")
plt.tight_layout(); plt.show()

## 3. Obstacle barrier

One or more circular obstacles $\{(c_o, r_o)\}$. The safe set is

$$K = \{\tau : \|z_t - c_o\|^2 \ge (r_o + m)^2 \ \forall t, o\}$$

for safety margin $m \ge 0$. We use a **soft-min** over waypoints and obstacles so $\nabla_\tau h$ is smooth (sharp $\min$ would give zero gradient almost everywhere):

$$h(\tau) = -\tfrac{1}{\beta} \log \sum_{t, o} \exp\!\left( -\beta \big( \|z_t - c_o\|^2 - (r_o + m)^2 \big) \right)$$

$h > 0$ means all waypoints are outside all obstacles (with margin). This is the geometric analog of MNIST's LSE-margin barrier.

In [ ]:
## A test scene with two obstacles
OBSTACLES = torch.tensor([
    [0.0,  0.8,  0.7],     # center_x, center_y, radius
    [0.0, -0.8,  0.7],
], device=device)
MARGIN = 0.1
BETA_H = 8.0   # soft-min sharpness

def obstacle_barrier(tau, obstacles=OBSTACLES, margin=MARGIN, beta=BETA_H):
    """h(tau) = -1/β · log Σ exp(-β (||z_t - c_o||² - (r_o+m)²))
       Shape: tau (B, T, 2) → h (B,). h>0 = safe."""
    B, T_, _ = tau.shape
    c = obstacles[:, :2]            # (O, 2)
    r = obstacles[:, 2]              # (O,)
    diff = tau.unsqueeze(2) - c.view(1, 1, -1, 2)        # (B, T, O, 2)
    d2 = (diff ** 2).sum(-1)                              # (B, T, O)
    margin_d2 = (r + margin) ** 2                         # (O,)
    slack = d2 - margin_d2.view(1, 1, -1)                 # (B, T, O); >0 means safe
    slack_flat = slack.view(B, -1)
    return -(1.0 / beta) * torch.logsumexp(-beta * slack_flat, dim=1)

## Plot the safe set
xs = torch.linspace(-4, 4, 80); ys = torch.linspace(-3, 3, 60)
gx, gy = torch.meshgrid(xs, ys, indexing="xy")
pts = torch.stack([gx.flatten(), gy.flatten()], dim=1)   # (N, 2)
# Build a fake trajectory of length 1 (one waypoint) per point to reuse the barrier
tau_grid = pts.view(-1, 1, 2).to(device)
with torch.no_grad():
    h_grid = obstacle_barrier(tau_grid).cpu().reshape(gx.shape)

fig, ax = plt.subplots(figsize=(7, 5))
cf = ax.contourf(gx, gy, h_grid, levels=20, cmap="RdYlGn", alpha=0.6)
ax.contour(gx, gy, h_grid, levels=[0], colors="k", linewidths=2)
for c in OBSTACLES.cpu():
    circle = plt.Circle((c[0], c[1]), c[2], fill=False, color="k", lw=2)
    ax.add_patch(circle)
    circle = plt.Circle((c[0], c[1]), c[2] + MARGIN, fill=False, color="k", ls="--", lw=1, alpha=0.5)
    ax.add_patch(circle)
ax.plot(*START.numpy(), "go", ms=12); ax.plot(*GOAL.numpy(), "r*", ms=14)
ax.set_aspect("equal"); ax.set_title("Obstacle barrier (per-waypoint slice; green=safe)")
plt.colorbar(cf, ax=ax, label="h"); plt.tight_layout(); plt.show()

## 4. CBF + CLF QP (closed-form batched, same as MNIST notebook)

Per sample:

$$\min_{u,\xi}\ \tfrac{1}{2}\|u - v_\theta(t,\tau)\|^2 + \tfrac{\rho}{2}\xi^2$$
$$\text{s.t.}\ \hat g^\top u \ge (-\partial_t h - \alpha h) / \|\nabla_\tau h\|, \qquad \nabla_\tau V^\top u \le -c V + \xi$$

Two CLF variants:
- **`relu`**: $V = \tfrac12 \text{relu}(-h)^2$ — colinear with the CBF (ablation).
- **`loglike`**: $s_t^\top u \ge s_t^\top v_t - \varepsilon - \xi$, with score from the OT-flow identity $s_t \approx (t v_t - x)/(1-t)$.

Solver uses the 4-active-set KKT enumeration (same code structure as MNIST v2). Note: here $\partial_t h = 0$ exactly because the obstacles don't move with $t$ in our setup.

In [ ]:
def score_from_velocity(t_scalar, x, v):
    """For x_t = (1-t)x_0 + t x_1, x_0 ~ N(0,I): s_t(x) ≈ (t·v - x)/(1-t)."""
    one_minus_t = max(1.0 - t_scalar, 1e-3)
    return (t_scalar * v - x) / one_minus_t


def cbf_signals(t_scalar, tau, *, beta_h=BETA_H, clf_kind="none", v_nom=None, eps_clf=0.0):
    """Returns torch tensors on device. tau (B, T, 2), shapes per-sample."""
    B = tau.size(0)
    tau_in = tau.detach().clone().requires_grad_(True)
    h = obstacle_barrier(tau_in, beta=beta_h)                   # (B,)
    grad = torch.autograd.grad(h.sum(), tau_in)[0].detach()     # (B, T, 2)

    g_flat = grad.reshape(B, -1)                                # (B, D)
    g_norm = g_flat.norm(dim=1).clamp(min=1e-8)
    g_hat  = g_flat / g_norm.unsqueeze(1)
    h_d    = h.detach()
    h_n    = h_d / g_norm

    # CLF terms
    if clf_kind == "none":
        V       = torch.zeros(B, device=tau.device)
        grad_V  = torch.zeros(B, g_flat.size(1), device=tau.device)
    elif clf_kind == "relu":
        relu_neg_h = torch.clamp(-h_d, min=0.0)
        V      = 0.5 * relu_neg_h ** 2
        grad_V = -(relu_neg_h.unsqueeze(1) * g_flat)
    elif clf_kind == "loglike":
        assert v_nom is not None
        s_t = score_from_velocity(t_scalar, tau.detach(), v_nom.detach())
        s_flat = s_t.reshape(B, -1)
        v_flat = v_nom.detach().reshape(B, -1)
        sv = (s_flat * v_flat).sum(dim=1)
        V      = -sv + eps_clf
        grad_V = -s_flat
    else:
        raise ValueError(clf_kind)

    return {"h": h_d, "h_n": h_n, "g_hat": g_hat, "g_norm": g_norm,
             "V": V, "grad_V": grad_V}


def solve_qp_batched(u_nom, sig, *, alpha=1.0, c_clf=2.0, rho=10.0, use_clf=True):
    """Closed-form batched QP (same as MNIST v2 — 4 KKT active sets per sample).
       min 0.5||u-u_nom||² + 0.5·rho·ξ²
       s.t. a_h·u ≥ b_h        (CBF, ||a_h||=1)
            a_V·u ≤ b_V + ξ    (CLF, soft)
    """
    B = u_nom.size(0)
    a_h = sig["g_hat"]; b_h = -alpha * sig["h_n"]
    a_V = sig["grad_V"]; b_V = -c_clf * sig["V"]

    u0 = u_nom
    cbf_ok_0 = (a_h * u0).sum(1) >= b_h
    clf_ok_0 = (a_V * u0).sum(1) <= b_V

    deficit_h = b_h - (a_h * u_nom).sum(1)
    u1 = u_nom + deficit_h.unsqueeze(1) * a_h
    clf_ok_1 = (a_V * u1).sum(1) <= b_V

    aV_nom = (a_V * u_nom).sum(1)
    aV_sq  = (a_V * a_V).sum(1)
    mu2 = ((aV_nom - b_V) / (aV_sq + 1.0 / rho)).clamp(min=0.0)
    u2 = u_nom - mu2.unsqueeze(1) * a_V
    cbf_ok_2 = (a_h * u2).sum(1) >= b_h

    aH_aV  = (a_h * a_V).sum(1)
    aH_nom = (a_h * u_nom).sum(1)
    r1 = b_h - aH_nom
    r2 = b_V - aV_nom
    A11 = torch.ones_like(aH_aV); A12 = -aH_aV
    A21 = aH_aV;                   A22 = -(aV_sq + 1.0 / rho)
    det = (A11 * A22 - A12 * A21)
    det = torch.where(det.abs() < 1e-10, torch.full_like(det, 1e-10), det)
    lam3 = (r1 * A22 - A12 * r2) / det
    mu3  = (A11 * r2 - r1 * A21) / det
    mu3  = mu3.clamp(min=0.0)
    u3   = u_nom + lam3.unsqueeze(1) * a_h - mu3.unsqueeze(1) * a_V

    def cost(u): return 0.5 * ((u - u_nom) ** 2).sum(1)

    cands = torch.stack([u0, u1, u2, u3], dim=1)
    feas  = torch.stack([
        cbf_ok_0 & clf_ok_0,
        torch.ones_like(cbf_ok_0) & clf_ok_1,
        cbf_ok_2,
        torch.ones_like(cbf_ok_0),
    ], dim=1)
    costs = torch.stack([cost(u0), cost(u1), cost(u2), cost(u3)], dim=1)
    costs = torch.where(feas, costs, torch.full_like(costs, float("inf")))
    pick  = costs.argmin(dim=1)
    u_out = cands[torch.arange(B, device=u_nom.device), pick]

    if not use_clf:
        feas_nc = torch.stack([cbf_ok_0, torch.ones_like(cbf_ok_0)], dim=1)
        cands_nc = torch.stack([u0, u1], dim=1)
        costs_nc = torch.stack([cost(u0), cost(u1)], dim=1)
        costs_nc = torch.where(feas_nc, costs_nc, torch.full_like(costs_nc, float("inf")))
        pick_nc = costs_nc.argmin(dim=1)
        u_out = cands_nc[torch.arange(B, device=u_nom.device), pick_nc]

    return u_out


def cbf_velocity(t_scalar, tau, flow_model, *,
                  beta_h=BETA_H, alpha=1.0, c_clf=2.0, rho=10.0,
                  clf_kind="none", eps_clf=0.0,
                  return_stats=False):
    B = tau.size(0)
    with torch.no_grad():
        t_in = torch.full((B,), float(t_scalar), device=tau.device)
        v_nom = flow_model(t_in, tau)

    sig = cbf_signals(t_scalar, tau, beta_h=beta_h, clf_kind=clf_kind,
                       v_nom=v_nom, eps_clf=eps_clf)
    u_flat = solve_qp_batched(
        v_nom.reshape(B, -1), sig,
        alpha=alpha, c_clf=c_clf, rho=rho, use_clf=(clf_kind != "none"),
    )
    u_safe = u_flat.view_as(v_nom)
    if return_stats:
        return u_safe, {
            "h_min":     float(sig["h"].min().item()),
            "h_mean":    float(sig["h"].mean().item()),
            "frac_unsafe": float((sig["h"] < 0).float().mean().item()),
            "corr_norm": float(((u_safe - v_nom).reshape(B, -1).norm(dim=1)).mean().item()),
            "V_mean":    float(sig["V"].mean().item()),
        }
    return u_safe

## 5. Samplers

In [ ]:
def sample_nominal_traj(flow_model, x0, num_steps=100):
    flow_model.eval()
    x = x0.clone(); dt = 1.0 / num_steps
    with torch.no_grad():
        for k in range(num_steps):
            t_in = torch.full((x.size(0),), k * dt, device=x.device)
            x = x + dt * flow_model(t_in, x)
    return x


def sample_cbf_traj(flow_model, x0, num_steps=100, *,
                     alpha=1.0, c_clf=2.0, rho=10.0,
                     clf_kind="none", eps_clf=0.0):
    flow_model.eval()
    x = x0.clone(); dt = 1.0 / num_steps
    stats_list = []
    for k in tqdm(range(num_steps), desc=f"cbf-{clf_kind}", leave=False):
        u, stats = cbf_velocity(k * dt, x, flow_model,
                                  alpha=alpha, c_clf=c_clf, rho=rho,
                                  clf_kind=clf_kind, eps_clf=eps_clf,
                                  return_stats=True)
        x = x + dt * u
        stats_list.append(stats)
    return x, stats_list


def sample_guided_traj(flow_model, x0, num_steps=100, *, lam=1.0):
    """Classifier-guidance baseline: u = u_nom + lam · ∇h (push toward safe)."""
    flow_model.eval()
    x = x0.clone(); dt = 1.0 / num_steps
    total_corr = []
    for k in tqdm(range(num_steps), desc=f"guide λ={lam}", leave=False):
        t_s = k * dt
        B = x.size(0)
        with torch.no_grad():
            t_in = torch.full((B,), t_s, device=x.device)
            u_nom = flow_model(t_in, x)
        x_r = x.detach().clone().requires_grad_(True)
        h = obstacle_barrier(x_r)
        g = torch.autograd.grad(h.sum(), x_r)[0].detach()
        u = u_nom + lam * g
        total_corr.append((lam * g).reshape(B, -1).norm(dim=1).mean().item())
        x = x + dt * u
    return x, float(np.mean(total_corr))

## 6. Run all methods + evaluate

**Metrics**
- **Collision rate**: fraction of trajectories with $h_\text{hard}(\tau) < 0$ at any waypoint (using `min`, not soft-min, for evaluation)
- **Goal error**: $\|z_T - \text{goal}\|$
- **Start error**: $\|z_0 - \text{start}\|$
- **Path length**: $\sum_t \|z_{t+1} - z_t\|$
- **Smoothness**: $\sum_t \|z_{t+1} - 2 z_t + z_{t-1}\|^2$
- **Correction norm**: mean $\|u_\text{safe} - u_\text{nom}\|$ per step

In [ ]:
@torch.no_grad()
def hard_collision(tau, obstacles=OBSTACLES, margin=MARGIN):
    """True if any waypoint is inside any obstacle (with margin). Shape: tau (B, T, 2) → (B,) bool."""
    c = obstacles[:, :2]; r = obstacles[:, 2]
    diff = tau.unsqueeze(2) - c.view(1, 1, -1, 2)
    d2 = (diff ** 2).sum(-1)                                 # (B, T, O)
    return ((d2 < (r + margin) ** 2)).any(dim=(1, 2))

@torch.no_grad()
def evaluate_traj(tau, name):
    coll = hard_collision(tau).float().mean().item()
    goal_err  = (tau[:, -1] - GOAL.to(tau.device)).norm(dim=1).mean().item()
    start_err = (tau[:, 0]  - START.to(tau.device)).norm(dim=1).mean().item()
    path_len  = (tau[:, 1:] - tau[:, :-1]).norm(dim=2).sum(dim=1).mean().item()
    smooth    = ((tau[:, 2:] - 2 * tau[:, 1:-1] + tau[:, :-2]) ** 2).sum(dim=(1, 2)).mean().item()
    print(f"{name:>28s}  coll={coll:5.3f}  goal_err={goal_err:5.3f}  start_err={start_err:5.3f}  "
          f"len={path_len:5.3f}  smooth={smooth:6.3f}")
    return {"name": name, "coll": coll, "goal_err": goal_err, "start_err": start_err,
             "path_len": path_len, "smooth": smooth, "counts": None}

In [ ]:
torch.manual_seed(42)
N_EVAL = 500
x0_eval = torch.randn(N_EVAL, T, 2, device=device)

# Nominal
x_nom = sample_nominal_traj(flow_model, x0_eval, num_steps=100)

# CBF
x_cbf, cbf_stats = sample_cbf_traj(flow_model, x0_eval, num_steps=100,
                                     alpha=1.0, clf_kind="none")

# CBF tighter (larger α = wider buffer interior)
x_cbf_tight, cbftight_stats = sample_cbf_traj(flow_model, x0_eval, num_steps=100,
                                                alpha=3.0, clf_kind="none")

# CBF + relu CLF (ablation, expected redundant)
x_cbf_relu, cbfrelu_stats = sample_cbf_traj(flow_model, x0_eval, num_steps=100,
                                              alpha=1.0, c_clf=2.0, rho=10.0,
                                              clf_kind="relu")

# CBF + loglike CLF
x_cbf_log, cbflog_stats = sample_cbf_traj(flow_model, x0_eval, num_steps=100,
                                            alpha=2.0, c_clf=1.0, rho=1.0, eps_clf=0.5,
                                            clf_kind="loglike")

# Classifier-guidance sweep
guide_results = {}
for lam in [0.5, 1.0, 2.0, 5.0]:
    x_g, corr = sample_guided_traj(flow_model, x0_eval, num_steps=100, lam=lam)
    guide_results[lam] = (x_g, corr)

In [ ]:
results = []
cbf_corr     = float(np.mean([s["corr_norm"] for s in cbf_stats]))
cbftight_corr= float(np.mean([s["corr_norm"] for s in cbftight_stats]))
relu_corr    = float(np.mean([s["corr_norm"] for s in cbfrelu_stats]))
log_corr     = float(np.mean([s["corr_norm"] for s in cbflog_stats]))

r = evaluate_traj(x_nom,        "nominal");                  r["corr"] = 0.0;          results.append(r)
r = evaluate_traj(x_cbf,        "CBF (α=1)");                r["corr"] = cbf_corr;     results.append(r)
r = evaluate_traj(x_cbf_tight,  "CBF (α=3)");                r["corr"] = cbftight_corr;results.append(r)
r = evaluate_traj(x_cbf_relu,   "CBF+relu-CLF");             r["corr"] = relu_corr;    results.append(r)
r = evaluate_traj(x_cbf_log,    "CBF+loglike-CLF");          r["corr"] = log_corr;     results.append(r)
for lam, (x_g, corr) in guide_results.items():
    r = evaluate_traj(x_g, f"guide λ={lam}"); r["corr"] = corr; results.append(r)

print("\n── correction norm ──")
for r in results:
    print(f"  {r['name']:>22s}  {r['corr']:.4f}")

## 7. Visualise trajectories

In [ ]:
def plot_trajs(ax, tau, title, color="steelblue", n_plot=80):
    ax.contourf(gx, gy, h_grid, levels=[h_grid.min(), 0, h_grid.max()],
                colors=["#fbb", "#cfc"], alpha=0.5)
    for c in OBSTACLES.cpu():
        ax.add_patch(plt.Circle((c[0], c[1]), c[2], fill=True, color="#b22", alpha=0.6))
    for k in range(min(n_plot, tau.size(0))):
        ax.plot(tau[k, :, 0].cpu(), tau[k, :, 1].cpu(), alpha=0.35, color=color, lw=1.0)
    ax.plot(*START.numpy(), "go", ms=10); ax.plot(*GOAL.numpy(), "r*", ms=12)
    ax.set_aspect("equal"); ax.set_title(title); ax.set_xlim(-4, 4); ax.set_ylim(-3, 3)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
plot_trajs(axes[0,0], x_nom,             "Nominal",             color="steelblue")
plot_trajs(axes[0,1], x_cbf,             "CBF (α=1)",           color="green")
plot_trajs(axes[0,2], x_cbf_tight,       "CBF (α=3)",           color="darkgreen")
plot_trajs(axes[1,0], x_cbf_log,         "CBF+loglike-CLF",     color="black")
plot_trajs(axes[1,1], guide_results[1.0][0], "clf-guide λ=1",   color="orange")
plot_trajs(axes[1,2], guide_results[5.0][0], "clf-guide λ=5",   color="darkorange")
plt.tight_layout(); plt.show()

## 8. Pareto fronts

In [ ]:
def find(name): return next(r for r in results if r["name"] == name)
nominal    = find("nominal")
cbf        = find("CBF (α=1)")
cbftight   = find("CBF (α=3)")
cbfrelu    = find("CBF+relu-CLF")
cbflog     = find("CBF+loglike-CLF")
guide      = [r for r in results if r["name"].startswith("guide")]

def scatter(ax, xfn, yfn, xlab, ylab, title):
    ax.scatter([xfn(r) for r in guide], [yfn(r) for r in guide],
               c="orange", s=70, label="clf-guidance", zorder=5)
    for r in guide:
        ax.annotate(r["name"].split("=")[-1], (xfn(r), yfn(r)),
                    textcoords="offset points", xytext=(5, 3), fontsize=8)
    ax.scatter([xfn(cbf)],     [yfn(cbf)],      c="green",    s=130, marker="*", label="CBF",              zorder=6)
    ax.scatter([xfn(cbftight)],[yfn(cbftight)], c="darkgreen",s=150, marker="*", label="CBF tight",        zorder=7)
    ax.scatter([xfn(cbfrelu)], [yfn(cbfrelu)],  c="blue",     s=130, marker="*", label="CBF+relu",         zorder=6)
    ax.scatter([xfn(cbflog)],  [yfn(cbflog)],   c="black",    s=160, marker="X", label="CBF+loglike",      zorder=8)
    ax.scatter([xfn(nominal)], [yfn(nominal)],  c="red",      s=80,  marker="D", label="nominal",          zorder=6)
    ax.set_xlabel(xlab); ax.set_ylabel(ylab); ax.set_title(title); ax.legend(fontsize=8)

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
scatter(axes[0], lambda r: r["corr"], lambda r: r["coll"],
        "correction norm", "collision rate", "safety vs distortion")
scatter(axes[1], lambda r: r["path_len"], lambda r: r["coll"],
        "path length", "collision rate", "safety vs path length")
scatter(axes[2], lambda r: r["goal_err"], lambda r: r["coll"],
        "goal error", "collision rate", "safety vs goal reach")
plt.tight_layout(); plt.show()

## 9. Diagnostics: $h$-trace, CBF active rate, MM Taylor residual

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].plot([s["h_min"]    for s in cbf_stats],      label="CBF",         color="green")
axes[0].plot([s["h_min"]    for s in cbftight_stats], label="CBF tight",   color="darkgreen")
axes[0].plot([s["h_min"]    for s in cbflog_stats],   label="CBF+loglike", color="black")
axes[0].axhline(0, color="k", ls="--"); axes[0].set_title("h_min across steps"); axes[0].legend()

axes[1].plot([s["frac_unsafe"] for s in cbf_stats],      label="CBF",         color="green")
axes[1].plot([s["frac_unsafe"] for s in cbftight_stats], label="CBF tight",   color="darkgreen")
axes[1].plot([s["frac_unsafe"] for s in cbflog_stats],   label="CBF+loglike", color="black")
axes[1].set_title("frac samples with h<0"); axes[1].set_xlabel("step"); axes[1].legend()

axes[2].plot([s["corr_norm"]   for s in cbf_stats],      label="CBF",         color="green")
axes[2].plot([s["corr_norm"]   for s in cbftight_stats], label="CBF tight",   color="darkgreen")
axes[2].plot([s["corr_norm"]   for s in cbflog_stats],   label="CBF+loglike", color="black")
axes[2].set_title("mean correction norm"); axes[2].set_xlabel("step"); axes[2].legend()
plt.tight_layout(); plt.show()

In [ ]:
## MM Taylor residual: compare h(t+dt, x+dt·v_nom) to its linear surrogate
torch.manual_seed(7)
x_r = torch.randn(16, T, 2, device=device)
dt = 1.0 / 100
residuals = []
for k in range(50):
    t_s = k * dt
    with torch.no_grad():
        t_in = torch.full((x_r.size(0),), t_s, device=device)
        v = flow_model(t_in, x_r)
    sig = cbf_signals(t_s, x_r, clf_kind="none", v_nom=v)
    grad_h = sig["g_hat"] * sig["g_norm"].unsqueeze(1)        # unnormalised
    h_t    = sig["h"]
    v_flat = v.reshape(v.size(0), -1)
    h_lin  = h_t + dt * (grad_h * v_flat).sum(dim=1)
    x_next = x_r + dt * v
    with torch.no_grad():
        h_true = obstacle_barrier(x_next)
    residuals.append((h_true - h_lin).mean().item())
    x_r = x_next

plt.figure(figsize=(7, 3))
plt.plot(residuals); plt.axhline(0, color="k", ls="--")
plt.title("MM Taylor residual: h_true − h_linear (per Euler step)")
plt.xlabel("step"); plt.ylabel("residual"); plt.tight_layout(); plt.show()
print(f"mean residual: {np.mean(residuals):.4f}  max |residual|: {np.max(np.abs(residuals)):.4f}")

## 10. Moving obstacle (time-varying $h$)

Demonstrates the full CBF form with $\partial_t h \neq 0$. The obstacle drifts upward as $t \to 1$. The CBF constraint becomes $\nabla_x h \cdot u + \partial_t h + \alpha h \ge 0$ — the $\partial_t h$ term anticipates obstacle motion.

In [ ]:
def moving_obstacle_barrier(t_scalar, tau, beta=BETA_H):
    """Single obstacle whose center drifts from (0, -1) at t=0 to (0, 1) at t=1."""
    c = torch.tensor([0.0, -1.0 + 2.0 * t_scalar], device=tau.device)
    r = 0.7
    diff = tau - c.view(1, 1, 2)                          # (B, T, 2)
    d2 = (diff ** 2).sum(-1)                              # (B, T)
    slack = d2 - (r + MARGIN) ** 2
    return -(1.0 / beta) * torch.logsumexp(-beta * slack, dim=1)


def cbf_signals_moving(t_scalar, tau, beta_h=BETA_H):
    """Same as cbf_signals but uses time-varying barrier and computes ∂h/∂t."""
    B = tau.size(0)
    tau_in = tau.detach().clone().requires_grad_(True)
    t_in   = torch.full((B,), float(t_scalar), device=tau.device, requires_grad=True)
    h_each = moving_obstacle_barrier(t_in[0], tau_in, beta=beta_h)
    grad_x = torch.autograd.grad(h_each.sum(), tau_in, retain_graph=True)[0].detach()
    grad_t = torch.autograd.grad(h_each.sum(), t_in)[0].detach()
    g_flat = grad_x.reshape(B, -1)
    g_norm = g_flat.norm(dim=1).clamp(min=1e-8)
    return {
        "h":     h_each.detach(),
        "h_n":   h_each.detach() / g_norm,
        "g_hat": g_flat / g_norm.unsqueeze(1),
        "dhdt_n": grad_t / g_norm,
        "g_norm": g_norm,
        "V":      torch.zeros(B, device=tau.device),
        "grad_V": torch.zeros(B, g_flat.size(1), device=tau.device),
    }


def sample_cbf_moving(flow_model, x0, num_steps=100, alpha=2.0):
    flow_model.eval()
    x = x0.clone(); dt = 1.0 / num_steps
    for k in tqdm(range(num_steps), desc="moving-cbf", leave=False):
        t_s = k * dt
        B = x.size(0)
        with torch.no_grad():
            t_in = torch.full((B,), t_s, device=x.device)
            v_nom = flow_model(t_in, x)
        sig = cbf_signals_moving(t_s, x)
        # include ∂h/∂t in the RHS
        rhs = -sig["dhdt_n"] - alpha * sig["h_n"]
        a_h = sig["g_hat"]
        v_flat = v_nom.reshape(B, -1)
        deficit = rhs - (a_h * v_flat).sum(1)
        u_flat = torch.where(deficit.unsqueeze(1) > 0,
                              v_flat + deficit.unsqueeze(1) * a_h,
                              v_flat)
        x = x + dt * u_flat.view_as(v_nom)
    return x

torch.manual_seed(0)
x0 = torch.randn(80, T, 2, device=device)
x_move_nom = sample_nominal_traj(flow_model, x0, num_steps=100)
x_move_cbf = sample_cbf_moving(flow_model, x0, num_steps=100, alpha=3.0)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, x_s, title in zip(axes, [x_move_nom, x_move_cbf], ["Nominal", "CBF (time-varying obstacle)"]):
    # show obstacle at t=0, 0.5, 1
    for tv, alpha_v in [(0.0, 0.3), (0.5, 0.5), (1.0, 0.7)]:
        c = (0.0, -1.0 + 2.0 * tv)
        ax.add_patch(plt.Circle(c, 0.7, color="#b22", alpha=alpha_v))
    for k in range(min(80, x_s.size(0))):
        ax.plot(x_s[k, :, 0].cpu(), x_s[k, :, 1].cpu(), alpha=0.3, color="steelblue", lw=1.0)
    ax.plot(*START.numpy(), "go", ms=10); ax.plot(*GOAL.numpy(), "r*", ms=12)
    ax.set_aspect("equal"); ax.set_xlim(-4, 4); ax.set_ylim(-3, 3); ax.set_title(title)
plt.tight_layout(); plt.show()

## Notes

### Why this is a cleaner story than MNIST
- $h$ is **geometric and exact** — no classifier training, no Tweedie, no kNN.
- $\nabla_x h$ is **closed-form** (no autograd through a CNN), one cheap backward pass through a soft-min.
- The collision metric is **objective**: a waypoint is either inside the obstacle or not.
- The CLF tradeoff is **visible in the plots**: CBF alone may produce kinks near the obstacle; CBF+loglike-CLF should keep trajectories smoother and more model-likely.

### What to look for
1. **CBF (α=1) vs CBF (α=3)**: tighter α should give zero collisions but at the cost of more correction.
2. **CBF+relu-CLF ≈ CBF**: relu CLF is colinear with the CBF safety direction; should not help.
3. **CBF+loglike-CLF**: should give similar collision rate as CBF tight, with **lower path length / better smoothness / lower goal error** — the model-likelihood term keeps the trajectory close to the nominal distribution.
4. **Classifier guidance λ-sweep**: as λ grows, collision drops but path length and goal error blow up — classic over-correction.
5. **Moving obstacle**: $\partial_t h$ term matters — without it, the CBF would be a step behind.

### Differences from the MNIST notebook
- No trained classifier (the geometry is the barrier).
- No prototypes, no feature space.
- Smaller flow model (MLP, not UNet); training takes ~30s.
- Built-in `∂h/∂t` term shown in the moving-obstacle section.